In [ ]:
import pandas as pd
import numpy as np
import ast

## data import

In [6]:
# firm data
firms_treated = pd.read_excel('../data/processed/clean_firm_data.xlsx')
# orbis
orbis_data = pd.read_csv('../data/processed/clean_orbis_base.csv', dtype={'nace_code': str})
orbis_match_treated = pd.read_excel('../data/interim/orbis_match_treated.xlsx')

empl_data = pd.read_excel('../data/raw/orbis/empl.xlsx', sheet_name='Risultati')
tot_ass_data = pd.read_excel('../data/raw/orbis/tot_assets.xlsx', sheet_name='Risultati')
op_rev_data = pd.read_excel('../data/raw/orbis/op_revenue.xlsx', sheet_name='Risultati')
rd_data = pd.read_excel('../data/raw/orbis/r&d_expenses.xlsx', sheet_name='Risultati')
intangible_data = pd.read_excel('../data/raw/orbis/intangible_assets.xlsx', sheet_name='Risultati')
ebit_data = pd.read_excel('../data/raw/orbis/ebit_margin.xlsx', sheet_name='Risultati')
ebitda_data = pd.read_excel('../data/raw/orbis/ebitda_margin.xlsx', sheet_name='Risultati')
longterm_debt_data = pd.read_excel('../data/raw/orbis/longterm_debt.xlsx', sheet_name='Risultati')
roe_data = pd.read_excel('../data/raw/orbis/roe.xlsx', sheet_name='Risultati')
current_ratio_data = pd.read_excel('../data/raw/orbis/current_ratio.xlsx', sheet_name='Risultati')
profit_marg_data = pd.read_excel('../data/raw/orbis/profit_margin.xlsx', sheet_name='Risultati')

orbis_pat = pd.read_csv('../data/interim/pat_orbis_counts.csv')

# regpat data
#regpat_data = pd.read_csv('../data/processed/pct_clean.csv')

# match df
match_df = pd.read_csv('../data/interim/firm_matches.csv')

## data cleaning and prep

In [7]:
firms_treated['start_year'] = firms_treated['start_date'].dt.year
firms_treated['end_year'] = firms_treated['end_date'].dt.year

# add bvd_id to firm data
firms_treated = firms_treated.merge(
    orbis_match_treated[['firm_name','bvd_id']],
    on = 'firm_name',
    how='left'
)

In [8]:
# drop extra rows from previous names
orbis_data = orbis_data.dropna(subset=['bvd_id'])

## firm data processing

In [12]:
# create first treat per firm
first_treat = (
    firms_treated
    .groupby('bvd_id')
    .agg(
        first_treat = ('start_year', 'min'),
        last_treat = ('end_year', 'max'),
        num_cartels_firm = ('cartel_id', 'nunique'),
        first_cart=('cartel_id', 'first') 
    )
    .reset_index()
)

first_treat['is_multicartel_firm'] = (first_treat['num_cartels_firm'] > 1).astype(int)

# time-varying treatment
temp = (
    firms_treated
    .assign(year=lambda x: x.apply(
        lambda r: list(range(r.start_year, r.end_year + 1)), axis=1))
    .explode('year')
)

firm_cartel_years = (
    temp.groupby(['bvd_id', 'year'])
    .agg(active_cartels_t=('cartel_id', 'nunique'))
    .reset_index()
)

firm_cartel_years['treat_t'] = (firm_cartel_years['active_cartels_t'] > 0).astype(int)

firm_cartel_years = firm_cartel_years.merge(
    orbis_data[['bvd_id', 'firm_id']],
    on='bvd_id',
    how='left'
).drop(columns='bvd_id')


In [13]:
firm_df = pd.merge(
    orbis_data[['firm_id', 'firm_name_preproc', 'ctry_code', 'nace_code', 'incorp_year', 'bvd_id']],
    first_treat, 
    on='bvd_id', 
    how='left')

"""firm_df = firm_df.merge(
    match_df[['firm_id']], 
    on='firm_id', 
    how='inner')"""

# create treatment and firm age indicators
firm_df['treat'] = firm_df['first_treat'].notna().astype(int)
# firm_df = firm_df.drop_duplicates(subset='firm_id')
print(firm_df[firm_df['treat'] == 1]['firm_id'].nunique())

131


### orbis controls

In [14]:
def reshape_orbis_data(df, id_col, col_prefix, value_name, new_id_name=None):
    
    df = df.copy()
    
    # Clean column names
    clean_cols = df.columns.str.strip().str.replace("\n", "_", regex=True).str.replace(" ", "_")
    df.columns = clean_cols
    
    # Map original names to cleaned names
    id_col_clean = id_col.strip().replace("\n", "_").replace(" ", "_")
    col_prefix_clean = col_prefix.strip().replace("\n", "_").replace(" ", "_")
    
    # Optionally rename the ID column
    if new_id_name:
        df.rename(columns={id_col_clean: new_id_name}, inplace=True)
        id_col_clean = new_id_name
    
    # Select relevant columns
    relevant_cols = [c for c in df.columns if c.startswith(col_prefix_clean)]
    
    # Melt to long format
    df_long = df.melt(
        id_vars=[id_col_clean],
        value_vars=relevant_cols,
        var_name='variable',
        value_name=value_name
    )
    
    # Extract year
    df_long['year'] = df_long['variable'].str.extract(r"_(\d{4})$").astype(int)
    
    # Drop helper column
    df_long = df_long.drop(columns=['variable'])
    
    return df_long


In [15]:
empl_long = reshape_orbis_data(empl_data, id_col='Numero BvD ID', col_prefix='Numero dipendenti ', value_name='empl', new_id_name='bvd_id')
op_rev_long = reshape_orbis_data(op_rev_data, id_col='Numero BvD ID', col_prefix='Totale valore della produzione migl EUR ', value_name='op_rev_th', new_id_name='bvd_id')
tot_ass_long = reshape_orbis_data(tot_ass_data, id_col='Numero BvD ID', col_prefix='Totale Attivo migl EUR ', value_name='tot_asset_th', new_id_name='bvd_id')
rd_long = reshape_orbis_data(rd_data, id_col='Numero BvD ID', col_prefix='Costi per ricerca e sviluppo migl EUR ', value_name='rd_th', new_id_name='bvd_id')
intangible_long = reshape_orbis_data(intangible_data, id_col='Numero BvD ID', col_prefix='Immobiliz. Immateriali migl EUR ', value_name='intangible_th', new_id_name='bvd_id')
ebit_long = reshape_orbis_data(ebit_data, id_col='Numero BvD ID', col_prefix='Margine EBIT ', value_name='ebit', new_id_name='bvd_id')
ebitda_long = reshape_orbis_data(ebitda_data, id_col='Numero BvD ID', col_prefix='Margine EBITDA ', value_name='ebitda', new_id_name='bvd_id')
longterm_debt_long = reshape_orbis_data(longterm_debt_data, id_col='Numero BvD ID', col_prefix='Indebitamento a lungo termine migl EUR ', value_name='longterm_debt_th', new_id_name='bvd_id')
roe_long = reshape_orbis_data(roe_data, id_col='Numero BvD ID', col_prefix='Redditività del capitale proprio (ROE) - Lordo ', value_name='roe', new_id_name='bvd_id')
current_ratio_long = reshape_orbis_data(current_ratio_data, id_col='Numero BvD ID', col_prefix='Indice di disponibilità ', value_name='current_ratio', new_id_name='bvd_id')
profit_marg_long = reshape_orbis_data(profit_marg_data, id_col='Numero BvD ID', col_prefix='Margine di profitto ', value_name='profit_marg', new_id_name='bvd_id')

In [16]:
orbis_controls = (
    empl_long
    .merge(op_rev_long, on=['bvd_id', 'year'], how='left')
    .merge(tot_ass_long, on=['bvd_id', 'year'], how='left')
    .merge(rd_long, on=['bvd_id', 'year'], how='left')
    .merge(intangible_long, on=['bvd_id', 'year'], how='left')
    .merge(ebit_long, on=['bvd_id', 'year'], how='left')
    .merge(ebitda_long, on=['bvd_id', 'year'], how='left')
    .merge(longterm_debt_long, on=['bvd_id', 'year'], how='left')
    .merge(roe_long, on=['bvd_id', 'year'], how='left')
    .merge(current_ratio_long, on=['bvd_id', 'year'], how='left')
    .merge(profit_marg_long, on=['bvd_id', 'year'], how='left')
)
# replace placeholder with nan
orbis_controls = orbis_controls.replace(['n.s.', 'n.d.'], np.nan)

C:\Users\nicol\AppData\Local\Temp\ipykernel_9072\2099436811.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orbis_controls = orbis_controls.replace(['n.s.', 'n.d.'], np.nan)


## patent data processing

In [ ]:
# merge pat data with firm match df
regpat_data = regpat_data.merge(
    match_df[['app_name_preproc', 'firm_id']],
    on='app_name_preproc',
    how='inner'
)
regpat_data['app_year'] = pd.to_datetime(regpat_data['app_year'], format='%Y', errors='coerce').dt.year
regpat_data['prio_year'] = pd.to_datetime(regpat_data['prio_year'], format='%Y', errors='coerce').dt.year
regpat_data['ipc_subclass'] = regpat_data['IPC'].str.split().str[0].str[:4]
regpat_data.rename(columns={'IPC': 'ipc'}, inplace=True)

In [ ]:
# count patent filings by priority year by firm
pat_prio = (
    regpat_data
    .groupby(['firm_id', 'prio_year'])
    .agg(
        pat_prio=('pct_nbr', 'count'),
        ipc_subclasses_prio=('ipc_subclass', 'nunique'),
        ipc_groups_prio=('ipc', 'nunique')
    )
    .reset_index()
    .rename(columns={'prio_year': 'year'})
)

# count patents by epo application year by firm
pat_app = (
    regpat_data
    .groupby(['firm_id', 'app_year'])
    .agg(
        pat_app = ('pct_nbr', 'count'),
        ipc_subclasses_app = ('ipc_subclass', 'nunique'),
        ipc_groups_app = ('ipc', 'nunique')
    )
    .reset_index()
    .rename(columns={'app_year': 'year'})
)

In [ ]:
# patent citations
appln_map = (
    regpat_data[['appln_id', 'firm_id']]
    .drop_duplicates()
)

chunks = pd.read_csv(
    '../data/raw/regpat/202501_PCT_CITATIONS.txt',
    sep='|',
    usecols=['Cited_Appln_id', 'Citing_pub_date'],
    dtype={
        'Cited_Appln_id': 'Int32',
        'Citing_pub_date': 'string'
    },
    chunksize=1_000_000
)

rows = []

for chunk in chunks:
    # extract citation year
    chunk['year'] = chunk['Citing_pub_date'].str[:4].astype(int)

    # add firm_id 
    chunk = chunk.merge(
        appln_map,
        left_on='Cited_Appln_id',
        right_on='appln_id',
        how='inner'   # keep only patents in regpat_data
    )

    # count firm–year citations directly
    rows.append(
        chunk.groupby(['firm_id', 'year']).size()
    )

forw_year = (
    pd.concat(rows)
      .groupby(level=[0, 1])
      .sum()
      .rename('forw_cit')
      .reset_index()
)

In [ ]:
pat_year = (
    pat_prio
    .merge(pat_app, on=['firm_id', 'year'], how='outer')
    .merge(forw_year, on=['firm_id', 'year'], how='outer')
)

## panel

In [17]:
# create panel structure
years = range(1996, 2025)

panel_df = (
    pd.DataFrame({'firm_id': firm_df['firm_id'].dropna().unique()})
    .assign(key=1)
    .merge(pd.DataFrame({'year': years, 'key':1}), on='key')
    .drop(columns='key')
)

### merge with firm data

In [ ]:
# firm
panel_df = panel_df.merge(firm_cartel_years, on=['firm_id','year'], how='left')
panel_df['active_cartels_t'] = panel_df['active_cartels_t'].fillna(0)
panel_df['treat_t'] = panel_df['treat_t'].fillna(0)
panel_df = panel_df.merge(firm_df, on='firm_id', how='left')
panel_df = panel_df.merge(orbis_controls, on=['bvd_id','year'], how='left')
panel_df.drop(['bvd_id'], axis=1, inplace=True)

In [ ]:
# firm
panel_df = panel_df.merge(firm_cartel_years, on=['firm_id','year'], how='left')
panel_df['active_cartels_t'] = panel_df['active_cartels_t'].fillna(0)
panel_df['treat_t'] = panel_df['treat_t'].fillna(0)
panel_df = panel_df.merge(firm_df, on='firm_id', how='left')
panel_df = panel_df.merge(orbis_controls, on=['bvd_id','year'], how='left')
panel_df = panel_df.merge(orbis_pat, on=['bvd_id','year'], how='left')
panel_df.drop(['bvd_id'], axis=1, inplace=True)

panel_df = panel_df.dropna(subset=['publ_number_count', 'prio_date_count', 'appl_number_count', 'ipc_code_count', 'frw_cit_count'], how='all')

### merge with patent data

In [ ]:
# patent data
panel_df = panel_df.merge(pat_year, on=['firm_id','year'], how='left')
patent_cols = ['pat_app','ipc_subclasses_app','ipc_groups_app','pat_prio',
               'ipc_subclasses_prio','ipc_groups_prio', 'forw_cit']
panel_df[patent_cols] = panel_df[patent_cols].fillna(0)

In [21]:
# string cols
string_cols = ['firm_name_preproc', 'ctry_code', 'nace_code']
for col in string_cols:
    panel_df[col] = panel_df[col].astype('string')

# year cols
year_cols = ['year', 'incorp_year', 'first_treat', 'last_treat']
for col in year_cols:
    panel_df[col] = pd.to_numeric(panel_df[col], errors='coerce').astype('Int64')

# remaining to numeric
for col in panel_df.columns:
    if col not in string_cols and col not in year_cols:
             panel_df[col] = pd.to_numeric(panel_df[col], errors='coerce')

# frequency encoding categorical vars
cat_vars= ['ctry_code', 'nace_code']
for col in cat_vars:
    freq = panel_df[col].value_counts(normalize=True)
    panel_df[col + '_freq'] = panel_df[col].map(freq)

### covariates

In [ ]:
panel_df['firm_age'] = panel_df['year'] - panel_df['incorp_year']
panel_df['nace2'] = panel_df['nace_code'].str[:2]


panel_df['log_empl'] = np.log(panel_df['empl'] + 1)
panel_df['log_op_rev'] = np.log(panel_df['op_rev_th'] + 1)
panel_df['log_asset'] = np.log(panel_df['tot_asset_th'] + 1)
panel_df['log_rd'] = np.log(panel_df['rd_th'] + 1)
panel_df['log_intang'] = np.log(panel_df['intangible_th'] + 1)
panel_df['log_long_debt'] = np.log(panel_df['longterm_debt_th'] + 1)
panel_df['log_ipc_subclasses_app'] = np.log(panel_df['ipc_subclasses_app'] + 1)


# lag covariates 
cols = ['log_empl', 'log_op_rev', 'log_asset', 'log_rd', 
              'log_intang', 'log_long_debt']

for var in cols:
    panel_df[f'{var}_lag1'] = panel_df.groupby('firm_id')[var].shift(1)

# ihs transform
panel_df['ihs_pat_prio'] = np.log(panel_df['pat_prio'] + np.sqrt(panel_df['pat_prio']**2 + 1))
panel_df['ihs_pat_app'] = np.log(panel_df['pat_app'] + np.sqrt(panel_df['pat_app']**2 + 1))

# market share
industry_group = panel_df.groupby(['nace_code', 'year'])
panel_df['market_share'] = panel_df['op_rev_th'] / industry_group['op_rev_th'].transform('sum')

c:\Users\nicol\projects\eu_cartel_innov\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\nicol\projects\eu_cartel_innov\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\nicol\projects\eu_cartel_innov\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\nicol\projects\eu_cartel_innov\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [23]:
# calculate patent stock
mu = 0.15

panel_df = panel_df.sort_values(['firm_id', 'year'])
panel_df['pat_stock'] = 0.0

for firm_id, g in panel_df.groupby('firm_id'):
    years = g['year'].values
    pats = g['pat_app'].fillna(0).values
    idx = g.index

    for t in range(len(g)):
        panel_df.loc[idx[t], 'pat_stock'] = np.sum(
            pats[:t+1] * np.exp(-mu * (years[t] - years[:t+1]))
        )

panel_df['ihs_pat_stock'] = np.log(
    panel_df['pat_stock'] + np.sqrt(panel_df['pat_stock']**2 + 1)
)

### calculate event time

In [24]:
# event time relative to entry
panel_df['event_time_entry'] = panel_df['year'] - panel_df['first_treat']
# event time relative to exit
panel_df['event_time_exit'] = panel_df['year'] - panel_df['last_treat']
# post entryS
panel_df['post_entry'] = (panel_df['year'] >= panel_df['first_treat']).astype('Int64')
panel_df['post_entry'] = panel_df['post_entry'].fillna(0)
#post exit
panel_df['post_exit'] = (panel_df['year'] >= panel_df['last_treat']).astype('Int64')
panel_df['post_exit'] = panel_df['post_exit'].fillna(0)

### save panel

In [ ]:
panel_df.to_parquet('../data/processed/panel_df.parquet', engine='fastparquet', index=False)
panel_df.to_csv('../data/processed/panel_df.csv', index=False)

## data overview and statistics

In [ ]:
# summary of firms and treatment groups
tot_firms = panel_df['firm_id'].nunique()
num_treated_firms = panel_df.loc[panel_df['treat']==1, 'firm_id'].nunique()
num_control_firms = tot_firms - num_treated_firms
treated_share = num_treated_firms / tot_firms
num_cartels = firms_treated['cartel_id'].nunique()

print(f"total firms: {tot_firms}")
print(f"treated firms: {num_treated_firms}")
print(f"control firms: {num_control_firms}")
print(f"treated share: {treated_share:.4f}%")
print(f"number of cartels: {num_cartels}")

In [ ]:
# treated firms with at least one patent
treated_firms_with_patent = panel_df[(panel_df['treat'] == 1) & (panel_df['filed_pat'] > 0)]
num_treated_firms_with_patent = treated_firms_with_patent['firm_id'].nunique()

# non-treated firms with at least one patent
control_firms_with_patent = panel_df[(panel_df['treat'] == 0) & (panel_df['filed_pat'] > 0)]
num_control_firms_with_patent = control_firms_with_patent['firm_id'].nunique()

total_treated_firms = panel_df[panel_df['treat'] == 1]['firm_id'].nunique()
total_control_firms = panel_df[panel_df['treat'] == 0]['firm_id'].nunique()

# %
pct_treated_patent = (num_treated_firms_with_patent / total_treated_firms) * 100
pct_control_patent = (num_control_firms_with_patent / total_control_firms) * 100

# Print the results
print(f"treated firms with at least one patent: {num_treated_firms_with_patent} ({pct_treated_patent:.2f}%)")
print(f"control firms with at least one patent: {num_control_firms_with_patent} ({pct_control_patent:.2f}%)")

In [ ]:
# nace group stats
nace_table = (
    panel_df.drop_duplicates(['firm_id'])
      .groupby(['nace_2dig', 'treat'])['firm_id']
      .count()
      .unstack(fill_value=0)
      .rename(columns={0:'controls', 1:'treated'})
)

nace_table['ratio_controls_per_treated'] = (
    nace_table['controls'] / nace_table['treated'].replace(0, np.nan)
)

nace_table